In [ ]:
from typing import Optional
from ollama import chat # type: ignore
from pydantic import BaseModel, Field # type: ignore
import json
import tiktoken
import pandas as pd
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr
import pandas as pd

def nd(arr):
    return np.asarray(arr).reshape(-1)

def yex(ax):
    lims = [
        np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
        np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
    ]

    # now plot both limits against eachother
    ax.plot(lims, lims, c="k", alpha=0.75, zorder=0)
    ax.set(**{"aspect": "equal", "xlim": lims, "ylim": lims})
    return ax
fsize = 15
plt.rcParams.update({"font.size": fsize})
%config InlineBackend.figure_format = 'retina'


In [38]:
def load_evidence(fn, ds_name):
    df = pd.read_json(fn)
    # Unwrap the 'source' column (contains a dictionary) into separate columns
    source_df = df['source'].apply(pd.Series)
    extracted_df = df['extracted'].apply(pd.Series).add_prefix('extracted_')
    # derived_df = df['derived'].apply(pd.Series).add_prefix('derived_')
    # Combine the original DataFrame with the unwrapped columns
    # df = pd.concat([df.drop(columns=['source', 'extracted', 'derived']), source_df, extracted_df, derived_df], axis=1)
    df = pd.concat([df.drop(columns=['source', 'extracted']), source_df, extracted_df], axis=1)
    df["ds"] = ds_name
    del df["derived"]
    return df

import hashlib

def short_hash(s: str) -> str:
    """Generate a 7-character hash from a given string."""
    return hashlib.sha256(s.encode()).hexdigest()[:7]

# Example usage
print(short_hash("mystring"))  # Example output: 'aaf4c61'

bd3ff47


In [12]:
from pydantic import BaseModel, Field
from typing import List, Optional

# Permissive class object (allows for missing fields)
class MarkerGene(BaseModel):
    marker_gene_name: Optional[str] = Field(
        None, 
        description="The name of the marker gene that is associated with a specific cell type."
    )
    cell_type_name: Optional[str] = Field(
        None, 
        description="The name of the cell type for which this gene serves as a marker."
    )

class MarkerGeneList(BaseModel):
    genes: List[MarkerGene]  # A list of marker genes extracted from the input text

# Restrictive class object does not allow for missing fields
class MarkerGeneStrict(BaseModel):
    marker_gene_name: str = Field(
        ..., 
        description="The name of the marker gene that is associated with a specific cell type."
    )
    cell_type_name: str = Field(
        ..., 
        description="The name of the cell type for which this gene serves as a marker."
    )

class MarkerGeneListStrict(BaseModel):
    genes: List[MarkerGeneStrict]  # A list of marker genes extracted from the input text

DATA_MODELS = {
    "MarkerGeneListStrict": MarkerGeneListStrict,
    "MarkerGeneStrict": MarkerGeneStrict
}

In [5]:
def extract_genes(user_prompt, system_prompt, data_model, model="llama3.2"):
    response = chat(
        messages=[
            {
                'role': 'system',
                'content': system_prompt,
            },
            {
                'role': 'user',
                'content': user_prompt,
            }
        ],
        model = model,
        format = data_model.model_json_schema(),
        options={'temperature': 0},  # Make responses more deterministic

    )
    genes = data_model.model_validate_json(response.message.content)
    return genes.model_dump()["genes"]

In [6]:
system_prompt = """
You are an expert in genomics analyzing scientific literature to extract marker genes for different cell types. 

Your goal is to identify and structure marker gene data from the given text. For each marker gene mentioned, extract:
- The **gene name** (marker_gene_name).
- The **cell type** it is associated with (cell_type_name).

The data must be extracted as written in the text, without any modifications.

Return the results in **structured JSON format** with the following schema:
{
    "genes": [
        {
            "cell_type_name": "Neuron",
            "marker_gene_name": "GeneX",
        },
        ...
    ]
}
"""
user_prompt = """
After doublet removal and quality filtering, we considered a total of 197,721 cells (106,469 from PG and 91,252 from ING), 
identifying all cell types observed in human WAT (Fig. 2c, d, Supplementary Table 2) with the addition of 
distinct male and female epithelial populations (Dcdc2a+ and Erbb4+, respectively)
"""

# system_prompt = "You are a genomics researcher trying to discover different cell types and what genes they express, aka find the marker genes within the given sentence."

In [7]:
fns = glob.glob("../../data/*/evidence_human/evidence.json")

In [ ]:
dfs = []
for fn in fns:
    ds_name = fn.split("/")[-3]
    dfs.append(load_evidence(fn, ds_name))
df = pd.concat(dfs)

In [9]:
df.groupby("source_type").size()

source_type
image    2763
text     1067
dtype: int64

In [10]:
tdf = df.query("source_type == 'text'").copy()
uniq_src = tdf.source_rationale.unique()

In [114]:
LLMODELS = {
    "deepseek-r1": "deepseek-r1",
    "llama3.2": "llama3.2",
    "llama3.3": "llama3.3"
}

In [115]:
model_metadata = {
    "model_id" : LLMODELS["deepseek-r1"],
    "system_prompt": system_prompt,
    "system_prompt_hash": short_hash(system_prompt),
    "data_model": DATA_MODELS["MarkerGeneListStrict"].__name__
}

In [ ]:
%%time
genes = []
for idx, u in enumerate(uniq_src):
    if idx % 25 == 0:
        print(f"Processing {idx}/{len(uniq_src)}")
    
    lg = extract_genes(u, model_metadata["system_prompt"], DATA_MODELS[model_metadata["data_model"]], model=model_metadata["model_id"])
    
    if len(lg) == 0:
        continue
    
    eg = []
    for g in lg:
        tmp = {
            "extracted": {
                "organism": "homo_sapiens",
                "cell_type_label": g["cell_type_name"],
                "cell_source": None,
                "cell_state": None,
                "feature_name": g["marker_gene_name"],
                "feature_type": "gene",
            },
            "derived": {
                "organism": "homo_sapiens",
                "cell_type_id": None,
                "cell_type_label": g["cell_type_name"],
                "cell_source": None,
                "cell_state": None,
                "feature_name": g["marker_gene_name"],
                "feature_type": "gene",
                "feature_identifier": None,
                "feature_identifier_type": None,
            },
            "source": {
                "source_type": "text",
                "source_rationale": u,
                "source_id": "text"
            }
        }
        genes.append(tmp)


os.mkdir(folder_name)
folder_name = f"evidence_llm_{model_metadata['model_id']}_{model_metadata['data_model']}_{model_metadata['system_prompt_hash']}"
with open(os.path.join(folder_name, "model_metadata.json"), 'w') as f:
    json.dump(model_metadata, f, indent=4)

with open(os.path.join(folder_name, "evidence.json"), 'w') as f:
    json.dump(genes, f, indent=4)


Processing 0/299
Processing 25/299
Processing 50/299
Processing 75/299
Processing 100/299
Processing 125/299
Processing 150/299
Processing 175/299
Processing 200/299
Processing 225/299
Processing 250/299
Processing 275/299
CPU times: user 828 ms, sys: 256 ms, total: 1.08 s
Wall time: 20min 48s


In [3]:
folder_name

'evidence_llm_llama3.2_MarkerGeneListStrict_4efcc22'